In [1]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

# Load dataset
df = pd.read_csv("Train.csv")

# Split target and features
y = df.iloc[:, 0]   # First column is the target variable
X = df.iloc[:, 1:]  # Remaining columns are features

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Improved Hybrid Resampling (SMOTE + Undersampling)
smote = SMOTE(sampling_strategy=0.2, random_state=42)  # Increase minority class more
undersample = RandomUnderSampler(sampling_strategy=0.5, random_state=42)  # Reduce majority

X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
X_resampled, y_resampled = undersample.fit_resample(X_resampled, y_resampled)

# Define XGBoost model with hyperparameter tuning
xgb = XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=42)

param_grid = {
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "n_estimators": [100, 200, 300],
    "scale_pos_weight": [1, (y_train.value_counts()[0] / y_train.value_counts()[1])]
}

grid_search = GridSearchCV(xgb, param_grid, scoring="f1", cv=3, n_jobs=-1)
grid_search.fit(X_resampled, y_resampled)

best_xgb = grid_search.best_estimator_
model = best_xgb

# Predict Probabilities
y_pred_prob = best_xgb.predict_proba(X_test)[:, 1]

# Adjust Decision Threshold
threshold = 0.7  # Adjusting threshold to favor minority class
y_pred = (y_pred_prob >= threshold).astype(int)

# Evaluation Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_prob)

print(classification_report(y_test, y_pred))
print(f"Accuracy = {accuracy:.4f}")
print(f"Precision = {precision:.4f}")
print(f"Recall = {recall:.4f}")
print(f"F1 = {f1:.4f}")
print(f"ROC-AUC = {roc_auc:.4f}")

metrics_note = f"""Accuracy = {accuracy:.4f}
Precision = {precision:.4f}
Recall = {recall:.4f}
F1 = {f1:.4f}
ROC-AUC = {roc_auc:.4f}
"""

with open("metrics_note.txt", "w", encoding="utf-8") as f:
    f.write(metrics_note)

import joblib
joblib.dump(model, "bankruptcy_model.pkl")


              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1060
           1       0.40      0.45      0.42        31

    accuracy                           0.97      1091
   macro avg       0.69      0.72      0.70      1091
weighted avg       0.97      0.97      0.97      1091

Accuracy = 0.9652
Precision = 0.4000
Recall = 0.4516
F1 = 0.4242
ROC-AUC = 0.9280


['bankruptcy_model.pkl']